# Parameter Golf — Model Inference

Load a trained model checkpoint and generate text.

**Prerequisites**:
- A trained model (run `train_gpt.py` first)
- The experiment's `train_gpt.py` (for model class definitions)
- Tokenizer files in `data/tokenizers/`

In [ ]:
import sys
import os
import json
import io
import math
import torch
import torch.nn.functional as F
import numpy as np

EXPERIMENT_DIR = "records/phase3/exp03_ema_from-exp02"
MODEL_PATH = "final_model.pt"
TOKENIZER_PATH = "data/tokenizers/fineweb_1024_bpe.model"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Device: {DEVICE}")

## 1. Import model classes from training script

In [ ]:
# Import the training script to get model architecture
sys.path.insert(0, EXPERIMENT_DIR)
import train_gpt as tg

# Load hyperparameters
args = tg.Hyperparameters()
print(f"Model config:")
print(f"  vocab_size: {args.vocab_size}")
print(f"  num_layers: {args.num_layers} (unique: {args.unique_layers})")
print(f"  model_dim: {args.model_dim}")
print(f"  num_heads: {args.num_heads}, num_kv_heads: {args.num_kv_heads}")
print(f"  mlp_mult: {args.mlp_mult}")
print(f"  seq_len: {args.train_seq_len}")

## 2. Build model and load weights

In [ ]:
# Build model
model = tg.GPT(
    vocab_size=args.vocab_size,
    num_layers=args.num_layers,
    model_dim=args.model_dim,
    num_heads=args.num_heads,
    num_kv_heads=args.num_kv_heads,
    mlp_mult=args.mlp_mult,
    tie_embeddings=args.tie_embeddings,
    tied_embed_init_std=args.tied_embed_init_std,
    logit_softcap=args.logit_softcap,
    rope_base=args.rope_base,
    qk_gain_init=args.qk_gain_init,
    bigram_vocab_size=args.bigram_vocab_size,
    bigram_dim=args.bigram_dim,
    unique_layers=args.unique_layers,
    rope_frac=args.rope_frac,
)

# Load state dict
state_dict = torch.load(MODEL_PATH, map_location="cpu")
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters on {DEVICE}")

## 2b. (Alternative) Load from quantized artifact

Use this if you only have the quantized `.ptz` file (the submission artifact).

In [ ]:
## Load from quantized artifact (.ptz)
## Make sure you've run cell 2 (model build) first — we need the model structure as template

import zstandard  # pip install zstandard

QUANT_PATH = "final_model.int8.ptz"

with open(QUANT_PATH, "rb") as f:
    quant_blob = f.read()

# Decompress
decompressed = zstandard.ZstdDecompressor().decompress(quant_blob)
quant_state = torch.load(io.BytesIO(decompressed), map_location="cpu", weights_only=False)

# Get a template state dict for shape/dtype info
template_sd = {k: v.detach().cpu() for k, v in model.state_dict().items()}

# Dequantize
deq_state = tg.dequantize_mixed_int6(quant_state["w"], quant_state["m"], template_sd)

# Handle AWQ inverse scales — undo the column scaling applied before quantization
awq_inv_keys = [k for k in deq_state if k.startswith("_awq_inv_scale.")]
print(f"AWQ inverse scale keys found: {len(awq_inv_keys)}")
for inv_key in awq_inv_keys:
    inv_scale = deq_state.pop(inv_key).float()
    layer_name = inv_key.replace("_awq_inv_scale.", "")
    weight_key = layer_name + ".weight"
    if weight_key in deq_state:
        deq_state[weight_key] = (deq_state[weight_key].float() * inv_scale.unsqueeze(0)).to(template_sd[weight_key].dtype)
        print(f"  unscaled {weight_key}")

# Remove any remaining AWQ metadata keys
deq_state = {k: v for k, v in deq_state.items() if not k.startswith("_awq_")}

# Load into model
model.load_state_dict(deq_state, strict=True)
model = model.to(DEVICE).eval()
print(f"\nLoaded quantized model from {QUANT_PATH}")
print(f"Artifact size: {len(quant_blob):,} bytes ({len(quant_blob)/1024/1024:.2f} MB)")

## 3. Load tokenizer

In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.Load(TOKENIZER_PATH)

print(f"Tokenizer loaded: vocab_size={sp.GetPieceSize()}")
print(f"Example: 'Hello world' -> {sp.Encode('Hello world')}")
print(f"Decoded back: '{sp.Decode(sp.Encode('Hello world'))}'")

## 4. Generation utilities

In [ ]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9,
    device: str = "cuda",
    seq_len: int = 2048,
) -> str:
    """Generate text from a prompt using the trained model."""
    model.eval()
    
    # Tokenize prompt
    tokens = tokenizer.Encode(prompt)
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    
    generated = list(tokens)
    
    for _ in range(max_new_tokens):
        # Truncate to seq_len if needed
        if input_ids.shape[1] > seq_len:
            input_ids = input_ids[:, -seq_len:]
        
        # Forward pass — forward_logits already applies softcap
        with torch.autocast(device_type=device.split(':')[0], dtype=torch.bfloat16, enabled=(device != 'cpu')):
            logits = model.forward_logits(input_ids)  # [1, seq_len, vocab]
        
        # Get logits for last position (softcap already applied)
        next_logits = logits[0, -1, :].float()
        
        # Temperature
        if temperature > 0:
            next_logits = next_logits / temperature
        
        # Top-k filtering
        if top_k > 0:
            top_k_vals, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < top_k_vals[-1]] = float('-inf')
        
        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_logits[indices_to_remove] = float('-inf')
        
        # Sample
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        
        generated.append(next_token)
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
    
    return tokenizer.Decode(generated)


@torch.no_grad()
def compute_perplexity(model, tokenizer, text: str, device: str = "cuda", seq_len: int = 2048) -> float:
    """Compute perplexity of the model on a given text."""
    model.eval()
    tokens = tokenizer.Encode(text)
    if len(tokens) < 2:
        return float('inf')
    
    input_ids = torch.tensor([tokens[:seq_len]], dtype=torch.long, device=device)
    target_ids = torch.tensor([tokens[1:seq_len+1]], dtype=torch.long, device=device)
    
    # Trim to same length
    min_len = min(input_ids.shape[1], target_ids.shape[1])
    input_ids = input_ids[:, :min_len]
    target_ids = target_ids[:, :min_len]
    
    with torch.autocast(device_type=device.split(':')[0], dtype=torch.bfloat16, enabled=(device != 'cpu')):
        logits = model.forward_logits(input_ids)
    
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), target_ids.reshape(-1))
    return math.exp(loss.item())


print("Generation utilities loaded.")

## 5. Generate text

In [ ]:
# Generate from a prompt
prompt = "The history of artificial intelligence began"

output = generate(
    model, sp, prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
    device=DEVICE,
    seq_len=args.train_seq_len,
)

print(f"Prompt: {prompt}")
print(f"\nGenerated:\n{output}")

In [ ]:
# Try different prompts
prompts = [
    "In a small village by the sea, there lived",
    "The most important scientific discovery of the 21st century",
    "def fibonacci(n):\n",
    "Once upon a time",
]

for p in prompts:
    out = generate(model, sp, p, max_new_tokens=100, temperature=0.7, device=DEVICE, seq_len=args.train_seq_len)
    print(f"\n{'='*60}")
    print(f"Prompt: {p}")
    print(f"Output: {out}")

## 6. Compute perplexity

In [ ]:
test_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data.",
    "asdf jkl qwerty zxcv random gibberish text that should have high perplexity",
]

for text in test_texts:
    ppl = compute_perplexity(model, sp, text, device=DEVICE, seq_len=args.train_seq_len)
    print(f"Perplexity: {ppl:.2f} | Text: {text[:60]}...")

## 7. Interactive generation

In [ ]:
# Interactive — change the prompt and re-run
prompt = "Today I learned that"  # <-- Edit this!
temperature = 0.8  # Lower = more deterministic, higher = more creative
max_tokens = 150

output = generate(
    model, sp, prompt,
    max_new_tokens=max_tokens,
    temperature=temperature,
    top_k=50,
    top_p=0.9,
    device=DEVICE,
    seq_len=args.train_seq_len,
)
print(output)